# Day05 下午个人项目：电商用户多维分析

**姓名：** 马晓明
**专题方向：** A / B / C / D / E

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。


## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。


## 任务0：个人配置与运行环境

In [2]:
# =========================
# 个人配置（修正版）
# =========================
from pathlib import Path
import pandas as pd
import numpy as np

STUDENT_NAME = "马晓明"
TOPIC = "A"          # 用户生命周期

# 后续配置保持不变
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        data_path = candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
        if data_path.exists():
            return candidate
    raise FileNotFoundError("未找到清洗后数据，请检查 output/day04_project/ecommerce_customer_cleaned.csv")

ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 马晓明
专题： A
输入数据： c:\ecommerce-user-analysis-seed\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： c:\ecommerce-user-analysis-seed\output\day05_analysis


In [3]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 马晓明
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [ ]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,6-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,6-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-6个月,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-6个月,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,object
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,object
Gender,object
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [5]:
# =========================
# 任务1：验收清洗后数据
# =========================

# 定义核心字段（此处使用所有字段，也可自行裁剪）
core_cols = df.columns.tolist()

# 构建验收表（单行多指标）
validation = pd.DataFrame({
    "指标": [
        "行数",
        "列数",
        "CustomerID重复数",
        "总缺失数",
        "Churn=0 (未流失)",
        "Churn=1 (已流失)"
    ],
    "值": [
        len(df),
        len(df.columns),
        df['CustomerID'].duplicated().sum(),
        df.isnull().sum().sum(),
        (df['Churn'] == 0).sum(),
        (df['Churn'] == 1).sum()
    ]
})

# 展示验收结果
print("【数据验收报告】")
display(validation)

# 可选：检查核心字段是否有缺失（详细）
missing_check = df[core_cols].isnull().sum()
if missing_check.sum() == 0:
    print("\n✅ 所有核心字段均无缺失值，数据完整。")
else:
    print("\n⚠️ 以下字段存在缺失值：")
    print(missing_check[missing_check > 0])

【数据验收报告】


,指标,值
0,行数,5630
1,列数,22
2,CustomerID重复数,0
3,总缺失数,0
4,Churn=0 (未流失),4682
5,Churn=1 (已流失),948



✅ 所有核心字段均无缺失值，数据完整。


In [6]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：一行数据代表一位用户（客户），包含该用户的人口统计特征（性别、婚姻状况、城市等级等）、平台使用行为（登录设备、App使用时长、注册设备数等）、购物偏好（偏好品类、支付方式）、满意度评分、投诉情况以及流失状态（Churn）等信息。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：CustomerID 是系统为每位用户分配的唯一标识符（ID），其数值大小（如 50001 vs 50002）仅用于区分不同用户，不具有数值上的大小、顺序或比例意义。ID 求平均值（如 mean(CustomerID)）得出的数字没有任何业务含义，不能代表任何用户群体的特征或中心趋势。在建模或分析中，应将其视为名义变量（分类变量），仅用于标识或关联，不作为特征参与统计或机器学习模型。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [7]:
# =========================
# 任务2：公共基础指标
# =========================

# 计算各项指标
overall_metrics = pd.DataFrame({
    "指标": [
        "用户数",
        "流失人数",
        "总体流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券使用次数",
        "平均返现",
        "平均App使用时长",
        "平均满意度",
        "平均距上次下单天数"
    ],
    "数值": [
        len(df),
        df['Churn'].sum(),
        f"{df['Churn'].mean():.2%}",
        f"{df['OrderCount'].mean():.2f}",
        f"{df['OrderCount'].median():.2f}",
        f"{df['CouponUsed'].mean():.2f}",
        f"{df['CashbackAmount'].mean():.2f}",
        f"{df['HourSpendOnApp'].mean():.2f}",
        f"{df['SatisfactionScore'].mean():.2f}",
        f"{df['DaySinceLastOrder'].mean():.2f}"
    ]
})

# 展示结果
print("【公共基础指标】")
display(overall_metrics)

【公共基础指标】


,指标,数值
0,用户数,5630
1,流失人数,948
2,总体流失率,16.84%
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [9]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = df['Churn'].mean()   # 从数据中计算

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：当前样本共有 5,630名用户，总体流失率为 16.84%。用户平均订单数为 2.96，平均满意度评分为 3.07（满分5分），平均返现金额为 177.22，平均距上次下单天数为 4.46天。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [10]:
# =========================
# 任务3：单维专题分析
# =========================

# 当前专题 A，主分组字段
segment_field = "TenureGroup"

# 定义 TenureGroup 的自然顺序（用于排序）
tenure_order = ["0-6个月", "6-12个月", "12-24个月", "24-48个月", "48个月以上"]

# 使用 groupby + agg 完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=('CustomerID', 'count'),
    流失人数=('Churn', 'sum'),
    流失率=('Churn', 'mean'),
    平均订单数=('OrderCount', 'mean'),
    平均满意度=('SatisfactionScore', 'mean'),
    平均返现金额=('CashbackAmount', 'mean'),
    平均优惠券使用次数=('CouponUsed', 'mean')
).reset_index()

# 按 TenureGroup 的自然顺序排序
segment_analysis['TenureGroup'] = pd.Categorical(
    segment_analysis['TenureGroup'], 
    categories=tenure_order, 
    ordered=True
)
segment_analysis = segment_analysis.sort_values('TenureGroup').reset_index(drop=True)

# 展示结果
print("【单维专题分析：不同 TenureGroup 的用户行为】")
display(segment_analysis)

# 可选：导出该表
segment_analysis.to_csv(OUTPUT_DIR / "segment_analysis.csv", index=False, encoding='utf-8-sig')

【单维专题分析：不同 TenureGroup 的用户行为】


,TenureGroup,用户数,流失人数,流失率,平均订单数,平均满意度,平均返现金额,平均优惠券使用次数
0,0-6个月,1967,689,0.35,2.47,3.13,158.79,1.55
1,6-12个月,1585,157,0.10,2.66,2.99,161.48,1.57
2,12-24个月,1574,102,0.06,3.64,3.06,200.72,1.97
3,24-48个月,500,0,0.00,3.70,3.08,225.29,2.04
4,48个月以上,4,0,0.00,2.00,2.00,226.38,1.25


In [12]:
# 检查点3：单维专题分析

# =========================
# 检查点3：单维专题分析
# =========================

# 确保 topic_fields 已定义（防止遗漏）
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：不同生命周期阶段的用户，在流失率、订单行为和满意度上存在怎样的差异？哪个阶段的用户最值得关注？

**数据现象：**

> TODO：群体：0-6个月（新用户）【用户数：1,967人（占全体的34.9%）流失率：35.0%； 平均订单数：2.47；平均满意度：3.13；平均返现金额：158.79】；群体：6-12个月（早期用户）【用户数：1,585人；流失率：9.9%；平均订单数：2.66；平均满意度：2.99；平均返现金额：161.48】群体：12-24个月（中期用户）【用户数：1,574人；流失率：6.5%；平均订单数：3.64；平均满意度：3.06；平均返现金额：200.72】群体：24-48个月（长期用户）【用户数：500人；流失率：0.0%；平均订单数：3.70；平均满意度：3.08；平均返现金额：225.29】群体：48个月以上（超长期用户）【用户数：仅4人（样本量极小）；流失率：0.0%；平均订单数：2.00；平均满意度：2.00；平均返现金额：226.3】

**可能解释：**

> TODO：新用户（0-6个月）流失率高达35%，可能与该阶段用户对平台的服务体验、商品质量或物流速度尚未建立信任有关。值得关注的是，该群体的用户数最多（1,967人），说明拉新渠道有效但留存能力不足。6-12个月用户流失率骤降至9.9%，可能表明用户经过初期适应后，若使用习惯已建立，则留存意愿显著提升。中期用户（12-24个月）订单数和返现金额明显增长，可能因平台忠诚度积累或优惠券刺激，需进一步验证是否为因果关系。24-48个月用户流失率为0，但需结合订单数和返现金额来看——该类用户贡献较高价值，值得关注其是否因长期满意度而留存，或受“沉默”样本影响。48个月以上用户仅有4人，结论不具有统计代表性，不宜过度解读。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [15]:

# 允许的双维字段
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}

# 选择两个不同维度
dim_1 = "TenureGroup"
dim_2 = "Complain"

# 确保维度有效且不同
assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "所选维度必须来自允许列表"
assert dim_1 != dim_2, "两个维度必须不同"

# 使用原始字段分组
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=('CustomerID', 'count'),
    流失人数=('Churn', 'sum'),
    流失率=('Churn', 'mean'),
    平均订单数=('OrderCount', 'mean')
).reset_index()

# 添加可读标签列（用于展示，但不影响检查点）
cross_analysis['Complain_label'] = cross_analysis['Complain'].map({0: '未投诉', 1: '已投诉'})

# 新增“样本提示”列：用户数<30 标记为“小样本”，否则“可观察”
cross_analysis['样本提示'] = cross_analysis['用户数'].apply(
    lambda x: '小样本' if x < 30 else '可观察'
)

# 按 TenureGroup 自然顺序排序
tenure_order = ["0-6个月", "6-12个月", "12-24个月", "24-48个月", "48个月以上"]
cross_analysis['TenureGroup'] = pd.Categorical(
    cross_analysis['TenureGroup'], 
    categories=tenure_order, 
    ordered=True
)
cross_analysis = cross_analysis.sort_values(['TenureGroup', dim_2]).reset_index(drop=True)

# 展示结果
print("【双维度交叉分析：TenureGroup × Complain】")
display(cross_analysis[['TenureGroup', 'Complain_label', '用户数', '流失人数', '流失率', '平均订单数', '样本提示']])

# 导出报表（包含原始 Complain 列）
cross_analysis.to_csv(OUTPUT_DIR / "cross_analysis.csv", index=False, encoding='utf-8-sig')
print(f"\n交叉分析报表已保存至：{OUTPUT_DIR / 'cross_analysis.csv'}")

【双维度交叉分析：TenureGroup × Complain】


,TenureGroup,Complain_label,用户数,流失人数,流失率,平均订单数,样本提示
0,0-6个月,未投诉,1341,320,0.24,2.42,可观察
1,0-6个月,已投诉,626,369,0.59,2.59,可观察
2,6-12个月,未投诉,1192,74,0.06,2.66,可观察
3,6-12个月,已投诉,393,83,0.21,2.66,可观察
4,12-24个月,未投诉,1135,46,0.04,3.79,可观察
5,12-24个月,已投诉,439,56,0.13,3.27,可观察
6,24-48个月,未投诉,356,0,0.00,3.82,可观察
7,24-48个月,已投诉,144,0,0.00,3.40,可观察
8,48个月以上,未投诉,2,0,0.00,2.50,小样本
9,48个月以上,已投诉,2,0,0.00,1.50,小样本



交叉分析报表已保存至：c:\ecommerce-user-analysis-seed\output\day05_analysis\cross_analysis.csv


In [16]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：0-6个月（新用户）且已投诉的用户。该组合的流失率高达 59%，显著高于同生命周期未投诉用户（24%）以及其他任何生命周期-投诉状态组合。同时，该组合的用户数为 626人，占新用户群体的 31.8%（626/1967），具有较大的业务影响面。

**该组合的用户数、流失率和比较对象：**
> TODO：组合：0-6个月 + 已投诉，用户数：626人；流失率：59% 。比较对象1：0-6个月 + 未投诉，用户数：1341人；流失率：24%；差值：已投诉用户流失率高出 35个百分点  。比较对象2：12-24个月 + 已投诉，用户数：439人；流失率：13%；差值：新用户投诉流失率高出 46个百分点 。

**是否存在小样本风险：**

> TODO：存在，但仅针对48个月以上用户组。48个月以上的未投诉用户仅2人，已投诉用户2人，合计4人，属于“小样本”，其流失率为0%不可作为可靠结论。其他所有组合的用户数均≥144人，属于“可观察”范围，结论具有统计意义。判断依据：根据预先设定的阈值（用户数<30为小样本），48个月以上两个组合（用户数=2）被标记为小样本，其余组合均为可观察。

**为什么不能直接写成因果结论：**

> TODO：因为交叉分析只能说明“关联性”，无法证明“因果性”。例如，新用户投诉率高、流失率高，可能两者都是第三个因素（如产品质量差、物流慢、客服响应不及时）共同作用的结果。用户可能因为体验差才投诉，也可能因为投诉后未得到满意解决才流失。也可能是投诉行为本身导致用户流失（如投诉后矛盾激化），也可能是流失前的“预警信号”。要验证因果关系，需要进一步做控制实验（如A/B测试）或使用因果推断方法（如工具变量、双重差分等），而不能仅凭分组统计差异就下结论。

## 任务5：输出统计报表（必做）

In [17]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [18]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(5, 8)
通过：cross_analysis.csv，形状为(10, 8)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1
在____用户中，____指标为____，与____相比____。对应证据表：____。

> TODO：在新用户（0-6个月）群体中，流失率为 35%，显著高于 6-12个月用户（10%） 和 12-24个月用户（6%）。对应证据表：segment_analysis（单维专题分析表）。

### 结论2

> TODO：投诉行为与高流失率存在强关联，尤其在 0-6个月新用户 中，已投诉用户的流失率为 59%，是未投诉用户流失率（24%）的 2.5倍。对应证据表：cross_analysis（双维交叉分析表）。



### 结论3

> TODO：用户订单数和返现金额随生命周期的延长而逐步提升，12-24个月用户的平均订单数为 3.64，平均返现为 200.72，均高于新用户（订单2.47，返现158.79）。对应证据表：segment_analysis（单维专题分析表）。

### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：当前数据无法支持因果推断：例如，投诉与流失之间的强关联可能源于其他未观测到的因素（如产品体验、客服质量、用户预期等），不能直接证明“投诉导致流失”或“流失导致投诉”。缺少订单金额和订单时间信息：无法分析用户价值（如客单价、GMV）或行为时间趋势（如流失前是否有预警信号）。48个月以上用户样本量极小（仅4人），结论不具代表性，不宜用于业务决策。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：建议：针对 0-6个月新用户且已投诉 的群体（626人，流失率59%），建立 投诉快速响应机制 和 首单体验关怀计划，例如：投诉后24小时内人工回访；投诉处理完成后提供小额优惠券（如5-10元无门槛券）；新用户首月内推送使用指南或专属客服入口。                                                                                                      验证方式：需要以下数据和方法验证效果：A/B测试数据：将新用户随机分为实验组（实施关怀计划）和对照组（常规流程），对比两组在投诉后的30天留存率；投诉处理时长数据：记录投诉从提交到解决的时间，分析处理时长与流失率的关系；投诉原因分类数据：识别高频投诉类型（如物流、质量、客服态度），针对性优化；时间序列数据：追踪用户投诉后7天、14天、30天的登录和下单行为，判断流失窗口期。

## 拓展任务（选做）

In [ ]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）

## 最终检查：GitHub提交前验收

In [19]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？